# Centering or Z-Score Data

## The two operations

**Centering only**: subtract the column mean from each column. Each column now has mean 0, but its variance is unchanged.

```python
X_centered = X - X.mean(axis=0)
```

**Z-scoring (standardization)**: subtract the column mean *and* divide by the column standard deviation. Each column now has mean 0 *and* variance 1.

```python
X_zscored = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
```

The two operations affect SVD/PCA differently because they change the relative variance of features.

## What centering does

After centering, the singular values reflect the **actual variance structure** of your data in its original units. A feature with naturally larger variance will dominate the top PCs.

When this is correct: when feature variances are *meaningful* on the same scale, and you *want* the dominant features to dominate the analysis.

## What z-scoring does

After z-scoring, every feature contributes equally to the total variance (each contributes exactly 1.0). The top PCs reflect **patterns of correlation**, not patterns of magnitude.

When this is correct: when features are on *different scales* and you don't want the largest-variance feature to drown everything else.

## The difference shown concretely

Imagine three columns of a matrix:

```
height_meters  weight_kg  income_dollars
        1.7        65          50000
        1.8        80          75000
        1.6        55          40000
```

Without z-scoring, `income_dollars` has variance ~$10^8$ while `height_meters` has variance ~0.01. SVD will produce a PC1 that's essentially "income" — heights and weights will be invisible.

With z-scoring, all three features have unit variance. PC1 captures whichever combination of all three features has the highest *correlation structure*. If income, height, and weight all correlate, PC1 will be a balanced mixture.

The two analyses give *qualitatively different* answers about the same data.

## When to z-score: use the correlation matrix

Z-scoring before SVD is mathematically equivalent to doing PCA on the **correlation matrix** instead of the covariance matrix. Both produce the same result.

Use z-scoring when:

1. **Features are in different units** (income in dollars, height in meters). Without z-scoring, units determine which PCs dominate, which is methodologically unsound.

2. **You care about relationships, not magnitudes**. Correlation-based PCA reveals structure regardless of feature scale.

3. **The feature with largest variance isn't more important** in any substantive sense. If your "most variable feature" is just measured in larger units, z-scoring corrects for that artifact.

4. **You're following a tradition that defaults to it**. Psychometrics, factor analysis, and many social sciences default to standardization. Reviewers in those fields expect it.

## When NOT to z-score: use the covariance matrix

Use centering only when:

1. **Features are already on a comparable scale**. This is your situation — CLR transformation puts every crime type on the same log-ratio scale. Z-scoring would *erase* genuine differences in compositional variance across crime types.

2. **Variance differences are meaningful**. If high-variance features are high-variance because they really matter more (not because of unit choice), z-scoring suppresses real signal.

3. **You want to detect "scale" patterns**. Sometimes one feature dominates for legitimate substantive reasons, and you want PC1 to reflect that dominance.

4. **You're doing compositional data analysis (CoDA)**. Aitchison's framework is built on the covariance of log-ratios. Z-scoring CLR data is not a standard CoDA operation and would distort the geometric meaning of compositional distances.

## Specifically for your project: do not z-score

Your data is already CLR-transformed. After CLR:

- All 26 crime types are in the same units (log-ratios relative to the geometric mean)
- Variance differences across crime types reflect **real compositional dynamics** — some crime types vary more month-to-month than others, and that's substantively meaningful
- Compositional distances are invariant under the CLR transformation, but z-scoring would destroy this invariance

If you z-scored your CLR data before PCA, you'd be telling the analysis "treat homicide and theft as equally variable per unit," which contradicts the compositional reality. Specifically:

- Homicide is a low-variance crime type in CLR space (relatively stable share month-to-month)
- Theft is a higher-variance type (its share fluctuates more)

This variance difference is *itself* a compositional finding. PC1 in your unstandardized analysis correctly weights more variable types more heavily. Z-scoring would erase this.

The CoDA literature is unambiguous on this point: standardization after log-ratio transformation is not standard practice. References like Aitchison's *The Statistical Analysis of Compositional Data* (1986) and Pawlowsky-Glahn et al.'s *Modeling and Analysis of Compositional Data* (2015) all advocate covariance-based PCA on CLR data.

## What centering actually does for SVD/PCA

You might be wondering: do you need to center at all?

**Yes**. Without centering, the first principal component will pick up the *mean* of your data rather than its variance. PC1 will essentially point toward your data cloud's center, and subsequent PCs will describe variance around the cloud's mean — but PC1 itself will be uninformative about the variance structure.

Sklearn's `PCA()` centers automatically. If you ran SVD manually with `np.linalg.svd`, you'd need to center first or you'd get the wrong answer.

There's one exception: **TruncatedSVD** in sklearn deliberately does *not* center, which is correct for sparse data (text/TF-IDF matrices) where centering would destroy sparsity. For dense data, always center.

## Quick decision tree

```
Are features in the same units?
├── Yes → are variance differences substantively meaningful?
│         ├── Yes → CENTER ONLY (covariance PCA)  ← your case
│         └── No  → Z-SCORE (correlation PCA)
└── No  → Z-SCORE (correlation PCA)
```

For CLR-transformed compositional data: **center only**. Don't z-score.

## Verifying with a quick sanity check on your data

If you're curious, you can compare what z-scoring would do to your analysis:

```python
import numpy as np
from sklearn.decomposition import PCA

# Your current approach: center only
pca_centered = PCA().fit(clr_data.values)

# Z-scored alternative
clr_zscored = (clr_data.values - clr_data.values.mean(axis=0)) / clr_data.values.std(axis=0, ddof=1)
pca_zscored = PCA().fit(clr_zscored)

print("Centered top 5 variance ratios:", pca_centered.explained_variance_ratio_[:5].round(3))
print("Z-scored top 5 variance ratios:", pca_zscored.explained_variance_ratio_[:5].round(3))

# Compare PC1 loadings
pc1_centered = pca_centered.components_[0]
pc1_zscored = pca_zscored.components_[0]
cos_sim = np.dot(pc1_centered, pc1_zscored) / (np.linalg.norm(pc1_centered) * np.linalg.norm(pc1_zscored))
print(f"PC1 cosine similarity (centered vs z-scored): {cos_sim:.4f}")
```

If you run this, expect:

- Variance ratios differ noticeably (z-scored will likely have a flatter spectrum because no single feature can dominate)
- PC1 cosine similarity probably falls in 0.5-0.9 — meaningfully different but with shared structure
- The substantive interpretation of PC1 will likely shift

The "right" answer for compositional data is the centered version. The z-scored version is *also* a valid PCA, but of a different object (the correlation matrix of CLR data) that doesn't have a clean interpretation in compositional terms.

## TL;DR

**For your project**: center, don't z-score. Sklearn's `PCA()` does this automatically. CLR transformation has already put your features on a common scale, so z-scoring would undo information about which crime types are genuinely more variable — and that variance information is what your structural break analysis is built on.

**In general**: z-score when features are in different units; center-only when they're on a comparable scale and variance differences are substantively meaningful. The decision hinges on whether you want PCA to reveal "which features dominate by sheer magnitude" (centered) or "which features correlate" (z-scored).

# SVD

## What SVD actually does

Singular Value Decomposition factors any real $m \times n$ matrix $A$ into:

$$A = U \Sigma V^T$$

where:
- $U$ is $m \times m$, columns orthonormal (rotations in row space)
- $\Sigma$ is $m \times n$ diagonal, non-negative entries sorted descending (the "stretches")
- $V^T$ is $n \times n$, rows orthonormal (rotations in column space)

In one sentence: SVD says **any linear transformation is a rotation, then a stretch along axes, then another rotation**. The singular values are the stretches; $U$ and $V$ tell you which rotations.

This generality is why SVD shows up everywhere. Almost any matrix problem can be reformulated through SVD.

## The applications, ranked by how often you'll use them

### 1. PCA — what you've been doing

Already covered, but for completeness: PCA is SVD on a centered data matrix. The right singular vectors ($V$) are the principal directions, and $U \Sigma$ gives the coordinates. This is by far the most common SVD application in data science.

You've now done this enough to understand it intimately.

### 2. Low-rank matrix approximation (the foundational use)

The Eckart-Young theorem (1936): **the best rank-$k$ approximation to $A$ in Frobenius or spectral norm is the truncated SVD**. Specifically:

$$A_k = U_{:, :k} \Sigma_{:k, :k} V_{:k, :}^T$$

keeps only the top $k$ singular values and corresponding vectors. No other rank-$k$ matrix is closer to $A$.

**Why this matters**: most "real-world" matrices have a sharp dropoff in singular values — a few large ones, then a long tail of small ones. The small ones often represent noise. So $A_k$ for moderate $k$ captures the signal and discards the noise.

Concrete uses:
- **Image compression**: an image is a matrix of pixel values. Truncated SVD with $k = 50$ on a 1000×1000 image stores 1000×50 + 50 + 50×1000 = 100,050 numbers instead of 1,000,000. Often visually indistinguishable from the original. This is a teaching example, not production compression (JPEG is better), but it shows the principle.
- **Denoising**: a noisy data matrix has its signal concentrated in the top few singular values. Reconstructing from only those values strips the noise out.

```python
import numpy as np
A = np.random.randn(100, 50)
U, s, Vt = np.linalg.svd(A, full_matrices=False)

# Keep top 5 components
k = 5
A_approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
# A_approx is the best rank-5 approximation to A
```

### 3. Solving linear systems and least squares

For overdetermined systems $A\mathbf{x} = \mathbf{b}$ (more equations than unknowns), the least-squares solution minimizes $\|A\mathbf{x} - \mathbf{b}\|^2$. SVD gives this directly:

$$\mathbf{x}^* = V \Sigma^+ U^T \mathbf{b}$$

where $\Sigma^+$ inverts nonzero singular values and zeros the rest. This is the **Moore-Penrose pseudoinverse**, and it's what `np.linalg.lstsq` uses under the hood.

**Why SVD matters here vs. normal equations**: solving $A^T A \mathbf{x} = A^T \mathbf{b}$ directly squares the condition number of the problem, amplifying numerical errors. SVD handles ill-conditioned matrices more robustly because it works with $A$ directly. For a near-singular matrix, SVD lets you set tiny singular values to zero (regularize), avoiding division-by-near-zero issues.

This is why scientific software almost always uses SVD-based solvers for least squares despite the higher computational cost.

### 4. Computing matrix rank, condition number, null space

- **Rank**: number of singular values above some tolerance. Easy and reliable, even when float arithmetic obscures exact ranks.
- **Condition number**: $\kappa(A) = s_1 / s_{\min}$. Quantifies how much numerical errors get amplified when you solve $A\mathbf{x} = \mathbf{b}$. A condition number of $10^{12}$ means you can lose 12 digits of precision.
- **Null space**: columns of $V$ corresponding to zero singular values span the null space. Useful for finding solutions to $A\mathbf{x} = 0$.

```python
A = some_matrix
U, s, Vt = np.linalg.svd(A)
rank = np.sum(s > 1e-10)
cond = s[0] / s[s > 1e-10][-1]
null_space_basis = Vt[rank:, :].T   # columns span null space
```

### 5. Recommender systems / collaborative filtering

The classic example: a user-item rating matrix is mostly empty (most users haven't rated most items). Low-rank SVD-style decomposition fills in plausible values.

If $A_{u, i}$ is "rating of user $u$ for item $i$," and you assume the matrix has approximate rank $k$, you can decompose:

$$A \approx U_k \Sigma_k V_k^T$$

where rows of $U_k$ are "user features" and rows of $V_k$ are "item features." Predicted rating for an unseen $(u, i)$ pair: just compute that entry of the rank-$k$ reconstruction.

The Netflix Prize (2009) was largely won with techniques in this family, particularly **SVD++** and matrix factorization variants that handle implicit feedback. Modern recommenders (deep learning, neural collaborative filtering) have extended this, but the SVD framing is still foundational.

For your work, this is less directly relevant unless you ever build recommender-style models. But knowing the framing helps when reading any paper involving "matrix factorization."

### 6. Latent Semantic Analysis (LSA) for text

Build a term-document matrix: rows are words, columns are documents, entries are TF-IDF weights. Apply SVD. The top $k$ singular vectors define a "semantic space" in which similar documents and words sit close together regardless of exact word overlap.

LSA was the original method behind early search engines and document similarity systems. It's been mostly superseded by word embeddings (Word2Vec, GloVe) and transformers (BERT etc.), but the SVD-of-text-matrix approach is still the simplest baseline for "topic modeling" and is sometimes used in production for its interpretability and speed.

### 7. Image processing — beyond compression

- **Eigenfaces** (Turk & Pentland, 1991): apply SVD to a matrix where each row is a flattened face image. Top singular vectors are "eigenfaces" — basis face images. Any face is approximated as a linear combination. Used for face recognition before deep learning.
- **Background subtraction in video**: stack video frames into a tall matrix. The background is approximately rank-1 (constant across time), so the dominant SVD component recovers it; subtracting gives moving foreground objects.
- **Image inpainting**: missing pixels can be filled in by enforcing a low-rank structure on the matrix of pixel values.

### 8. Procrustes alignment — comparing two configurations

You have two configurations of points (e.g., PC1-3 coordinates from two different ε values) and you want to ask "are these the same up to rotation, scaling, and reflection?" SVD answers this directly via the **orthogonal Procrustes problem**:

$$\min_{R: R^T R = I} \|A - BR\|_F$$

The optimal $R$ comes from the SVD of $A^T B$:

$$A^T B = U \Sigma V^T \implies R^* = U V^T$$

Used for:
- **Comparing PCA fits across conditions** (this would have been an alternative to your subspace-angles diagnostic earlier)
- **Aligning protein structures** in computational biology
- **Shape analysis** in anatomy and morphometrics
- **Multi-omics integration**: aligning datasets with shared samples but different feature spaces

For your project, this is genuinely useful: when you fit PCA per era (pre-COVID, COVID, post-COVID), Procrustes alignment lets you ask "did the principal subspace rotate, and by how much?"

```python
from scipy.spatial import procrustes
mtx1, mtx2, disparity = procrustes(coords_pre_covid, coords_covid)
# disparity ranges from 0 (identical up to rotation) to ~1 (unrelated)
```

### 9. Numerical linear algebra primitive

SVD is used inside many other algorithms you might not realize:

- **`numpy.linalg.lstsq`**: SVD-based least squares
- **`numpy.linalg.pinv`**: SVD computes the Moore-Penrose pseudoinverse
- **`numpy.linalg.matrix_rank`**: counts singular values above tolerance
- **TruncatedSVD in sklearn**: SVD without centering, used for sparse matrices (text data) where centering would destroy sparsity
- **`scipy.linalg.subspace_angles`**: implemented via SVD of a matrix product (you used this earlier)

If you've used any of these, you've used SVD whether you knew it or not.

### 10. Deep learning — yes, really

A few places SVD shows up in modern deep learning:

- **Initialization**: orthogonal initialization of neural network weights uses SVD or QR to produce orthogonal matrices, which preserves variance through layers and improves gradient flow.
- **Model compression**: factoring large weight matrices into low-rank approximations via SVD shrinks model size with minimal accuracy loss. Used in mobile deployment.
- **LoRA (Low-Rank Adaptation)**: a recent fine-tuning technique for large language models. Instead of updating all weights of a pretrained model, you learn a low-rank update $\Delta W = AB$ where $A$ and $B$ are skinny. SVD of the desired weight change tells you the optimal low-rank factorization. This is in production at most LLM fine-tuning operations.
- **Spectral analysis of trained networks**: looking at singular values of weight matrices reveals how much information each layer is compressing, and is used as a diagnostic for training health.

### 11. Causal inference — synthetic controls and matrix completion

A relatively recent application: in causal inference with panel data, you have units × time periods of outcomes. Some cells are "treated" (you want to know the counterfactual). The synthetic controls and matrix completion literature (Athey et al., 2021) uses low-rank SVD-style decomposition to impute counterfactual outcomes:

> "What would Chicago crime composition have looked like in 2020-2022 if there had been no pandemic?"

You'd treat post-March-2020 Chicago crime data as "missing," fit a low-rank approximation using pre-COVID Chicago plus other cities, and the reconstruction gives you the counterfactual. Subtracting from observed gives the causal effect.

This is genuinely relevant to your project as a possible extension — though out of scope until you finish the structural break work.

### 12. Geometry: shape decomposition, fitting hyperplanes

The first column of $V$ in SVD gives the direction of greatest spread. The last column gives the direction of least spread, which equals the normal vector of the best-fit hyperplane through the data.

- **Total least squares**: regression that accounts for noise in both X and Y, solved by SVD.
- **Plane fitting in 3D point clouds**: smallest singular value's vector = plane normal.
- **Linear discriminant analysis**: a related decomposition (generalized eigenvalues, computed via SVD).

## Why SVD is so useful: three structural properties

The reason SVD shows up everywhere comes down to three properties:

**1. It always exists.** Eigenvalue decomposition only works for square matrices and may not exist (defective matrices have no full eigenbasis). SVD exists for every real matrix, square or rectangular, full-rank or not.

**2. It's numerically stable.** Computing SVD via standard algorithms (Jacobi, Golub-Kahan, randomized SVD) is robust to floating-point error. By contrast, computing eigenvalues of a non-symmetric matrix can be hairy.

**3. It generalizes the eigenvalue decomposition cleanly.** For symmetric positive-semidefinite matrices, SVD and eigenvalue decomposition coincide. For everything else, SVD provides a similar structural insight — directions of variance, magnitudes of stretching — without requiring symmetry.

## What you'll likely use in your project

Of these 12 applications, three are directly relevant to your work:

- **PCA** (#1): already in heavy use.
- **Procrustes alignment** (#8): comparing PC subspaces across eras when you fit per-era PCAs.
- **Low-rank approximation reasoning** (#2): the framework for thinking about reconstruction error and the noise floor in your singular value spectrum.

The rest are good to know about, but not on the critical path. If a referee asks, "Have you considered Procrustes-aligning your era-specific PCAs?", you'll know what they mean and how to do it. If someone mentions "low-rank approximation" or "Eckart-Young," you'll associate it with the truncated SVD.

## A useful mental model

If you remember nothing else, remember this: **SVD finds the most efficient orthonormal basis for representing the row and column structure of any matrix simultaneously.** Most data analysis problems can be framed as "find the low-dimensional structure in this matrix," and SVD is usually the first thing to try.

PCA is just one specific application of this idea. Once you see the pattern, you'll spot it in places you didn't expect — recommender systems, image processing, neural network compression, causal inference, and many others.

```
# ------------------------------------------------------------
# Compare singular value spectra across different ε values
# ------------------------------------------------------------
for eps, r in results.items():
    # Print the top 5 singular values for each epsilon setting
    print(f"ε={eps}: top-5 SV = {r['singular_values'][:5].round(2)}")

# ------------------------------------------------------------
# Compare PC1 loadings via cosine similarity
# ------------------------------------------------------------
# Extract PC1 loading vectors for ε = 0.02 and ε = 0.05
v1_002 = results[0.02]["loadings"][0]
v1_005 = results[0.05]["loadings"][0]

# Cosine similarity measures alignment between two vectors
cos_sim = np.dot(v1_002, v1_005) / (norm(v1_002) * norm(v1_005))
print(f"\nPC1 loading cosine similarity (ε=0.02 vs 0.05): {cos_sim:.4f}")

# ------------------------------------------------------------
# Compare PC2 and PC3 loadings the same way
# ------------------------------------------------------------
v2_002 = results[0.02]["loadings"][1]
v2_005 = results[0.05]["loadings"][1]

v3_002 = results[0.02]["loadings"][2]
v3_005 = results[0.05]["loadings"][2]

# Cosine similarity measures alignment between two vectors
cos_pc2 = np.dot(v2_002, v2_005) / (norm(v2_002) * norm(v2_005))
cos_pc3 = np.dot(v3_002, v3_005) / (norm(v3_002) * norm(v3_005))

print(f"\nPC2 cosine similarity: {cos_pc2:.4f}")
print(f"PC3 cosine similarity: {cos_pc3:.4f}")

# ------------------------------------------------------------
# Subspace comparison using principal angles
# ------------------------------------------------------------
# Stack the first 3 loading vectors into matrices (3 × K)
V_002 = results[0.02]["loadings"][:3]   # First 3 PCs for ε=0.02
V_005 = results[0.05]["loadings"][:3]   # First 3 PCs for ε=0.05

# Compute principal angles between the two 3D subspaces
# Note: subspace_angles expects matrices with shape (features × components)
angles = subspace_angles(V_002.T, V_005.T)

# Convert to degrees for interpretability
print(f"\nPrincipal angles (degrees): {np.degrees(angles).round(2)}")
```